# LangChain para Dummies

> **Objetivo:** Aprender a usar LangChain para interactuar con LLMs de forma más estructurada y potente que con la API directa.

## ¿Por qué LangChain y no la API directa?

| API directa (`google-genai`) | LangChain |
|------------------------------|----------|
| Solo para Gemini | Funciona con OpenAI, Claude, Gemini, LLaMA... |
| Código diferente por proveedor | Código igual, solo cambias el modelo |
| Sin herramientas integradas | Tools, parsers, memory, agents... |
| Prompts como strings | Templates reutilizables con variables |

> **LangChain es el framework estándar de la industria para construir aplicaciones con LLMs.**

---

## 1. Instalar LangChain

`langchain-google-genai` es el paquete que conecta LangChain con los modelos de Google (Gemini/Gemma).

In [1]:
# -q = modo silencioso (quiet), no muestra logs de instalación
!pip install langchain-google-genai -q

## 2. Cargar la API Key

Mismo patrón que siempre: leemos la clave del archivo `.env` para no exponerla en el código.

In [2]:
from dotenv import load_dotenv
import os

# Carga las variables del archivo .env al entorno de Python
load_dotenv(dotenv_path="C:/Users/Oscar/OneDrive - FM4/Escritorio/EVOLVE/Data Science/EVOLVE/Victor_Prior_IA_Generativa/Proyecto_Modulo_IAGen/.env")

# Lee la variable de entorno
API_KEY = os.getenv("Gemini_API_Key_001")
print("✅ API Key cargada")

✅ API Key cargada


## 3. Crear el modelo (LLM)

En LangChain, el modelo es un objeto Python que abstrae la conexión con la API.

### Parámetro clave: `temperature`
Controla la **aleatoriedad** de las respuestas:
- `temperature=0.0` → respuestas deterministas y precisas (ideal para clasificación, código)
- `temperature=0.5` → equilibrio entre precisión y variedad
- `temperature=1.0` → respuestas más creativas e impredecibles (ideal para escritura creativa)

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

MODEL = "gemini-2.5-flash-lite"   # Modelo más rápido y económico de Gemini

# Crea el objeto LLM de LangChain
# Este objeto es el que usaremos en todas las cadenas (chains)
llm = ChatGoogleGenerativeAI(
    model=MODEL,
    temperature=0.5,            # Equilibrio creatividad/precisión
    google_api_key=API_KEY
)

print(f"✅ Modelo {MODEL} listo")

✅ Modelo gemini-2.5-flash-lite listo


## 4. Invocación Simple (`invoke`)

`invoke()` es la forma más básica: envías una pregunta y recibes una respuesta completa.

```
pregunta → llm.invoke() → AIMessage (objeto con .content)
```

> **Diferencia con la API directa:** LangChain devuelve un objeto `AIMessage`, no texto puro. El texto está en `.content`.

In [4]:
pregunta = "¿En que año llego el ser humano a la luna por primera vez?"
print("Pregunta: ", pregunta)

Pregunta:  ¿En que año llego el ser humano a la luna por primera vez?


In [5]:
# llm.invoke() envía el mensaje y espera la respuesta completa antes de devolverla
respuesta = llm.invoke(pregunta)

# respuesta es un AIMessage — accedemos al texto con .content
print("Respuesta del modelo: ", respuesta.content)

Respuesta del modelo:  El ser humano llegó a la Luna por primera vez en el año **1969**.


## 5. Stream (Respuesta en Tiempo Real)

`stream()` muestra la respuesta **token a token** según se genera, en lugar de esperar a que termine.

### ¿Cuándo usarlo?
- En chatbots o interfaces de usuario: da sensación de respuesta inmediata
- Cuando las respuestas son largas: el usuario ve que algo está pasando

### ¿Por qué el modelo responde así?
Los LLMs generan texto token a token (no frase a frase). `stream()` te deja ver ese proceso en tiempo real.

In [6]:
# stream() devuelve un iterador de "chunks" (fragmentos)
# Cada chunk es un pedazo de la respuesta según llega del servidor
# flush=True hace que cada fragmento se imprima inmediatamente (sin buffer)
for chunk in llm.stream("Why do parrots have colorful feathers?"):
    print(chunk.text, end="|", flush=True)

# El símbolo | entre palabras muestra los "chunks" de texto que llegan

## 6. Batch (Varias Preguntas a la Vez)

`batch()` envía múltiples preguntas **en paralelo** y devuelve todas las respuestas.

### ¿Cuándo usarlo?
- Procesar lotes de datos (miles de textos para clasificar)
- Más eficiente que hacer llamadas secuenciales una a una

In [7]:
# batch() recibe una lista de preguntas y devuelve una lista de AIMessages
responses = llm.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?"
])

# Mostramos solo los primeros 200 caracteres de cada respuesta
for i, response in enumerate(responses):
    print(f"Respuesta {i+1}:")
    print(response.content[:200] + "...")
    print("-" * 50)

## 7. Herramientas (Tools)

Las **Tools** permiten al modelo llamar a funciones Python que tú defines. El modelo decide cuándo usarlas.

### ¿Cómo funciona?
```
1. Defines una función Python con @tool
2. Conectas la función al modelo (bind_tools)
3. El modelo decide si necesita llamarla basándose en el prompt
4. Si la necesita, te devuelve la intención de llamarla (tú decides ejecutarla o no)
```

### ¿Para qué sirve?
Permite al modelo **"salir" del texto** y hacer cosas reales:
- Consultar una base de datos
- Buscar en internet
- Llamar a una API externa
- Ejecutar cálculos precisos

In [8]:
!pip install langchain -q

In [9]:
from langchain.tools import tool

In [10]:
# El decorador @tool convierte una función Python normal en una Tool de LangChain
# El docstring es CRÍTICO: el modelo lo lee para saber cuándo y cómo usar la función
@tool
def get_weather(location: str) -> str:
    """Get the weather at a location"""  # ← El modelo lee esto para decidir si usar la herramienta
    # En la realidad, aquí llamarías a una API de meteorología real
    return f"It is sunny in {location}"

In [11]:
# bind_tools() informa al modelo de qué herramientas tiene disponibles
# Devuelve un nuevo objeto LLM con las herramientas vinculadas
model_with_tools = llm.bind_tools([get_weather])

In [12]:
# El modelo detecta que la pregunta requiere conocer el tiempo → decide usar la herramienta
# IMPORTANTE: aquí el modelo NO devuelve texto de respuesta, devuelve una "intención" de llamar a la función
response = model_with_tools.invoke("What is the weather in Boston?")

In [13]:
# tool_calls contiene la lista de herramientas que el modelo quiere llamar
# Cada elemento tiene: name (nombre de la función) y args (argumentos)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Boston'}


## 8. Templates (Plantillas de Prompts)

Un **PromptTemplate** es un prompt con **variables** que se rellenan en tiempo de ejecución.

### ¿Por qué usar templates?
- Reutilizas el mismo prompt con diferentes valores
- Separas la lógica del prompt de los datos
- Más limpio y mantenible que concatenar strings

### Sintaxis:
```python
template = "Explica {concepto} para un {nivel}"
# {concepto} y {nivel} son variables que rellenarás al invocar
```

In [14]:
from langchain_core.prompts import PromptTemplate

# Las variables entre {} se sustituyen cuando invocas el template
template = """
Actua como un experto en {domain}.

Explica el siguiente concepto:
{concept}

A un nivel {nivel}

Usa ejemplos.
"""

# PromptTemplate.from_template() detecta automáticamente las variables entre {}
prompt = PromptTemplate.from_template(template)

In [15]:
# El operador | (pipe) encadena componentes: template → llm
# Esto es una "chain" (cadena): cada componente pasa su salida al siguiente
chain = prompt | llm

# invoke() rellena las variables del template y envía el prompt completo al modelo
response = chain.invoke({
    "domain": "machine learning",
    "concept": "overfitting",
    "nivel": "principiante"
})

print(response.content)

## 9. Anatomía de un Prompt Completo

Un prompt bien estructurado tiene 4 partes:

```
1. TAREA     → ¿Qué quieres que haga?
2. CONTEXTO  → Información relevante para la tarea
3. INSTRUCCIONES → Cómo debe hacerlo
4. FORMATO   → Cómo debe presentar la respuesta
```

### `SystemMessage` vs `HumanMessage`:
- **SystemMessage**: instrucciones de comportamiento para el modelo ("actúa como...")
- **HumanMessage**: el input del usuario
- **AIMessage**: la respuesta del modelo (también se puede incluir para dar contexto)

In [16]:
from langchain_core.messages import SystemMessage, HumanMessage

# ❌ Prompt vago — respuesta impredecible
human_basico = "Analiza este texto"

# ✅ Prompt estructurado — respuesta predecible y accionable
human_completo = """
TAREA: Analiza el sentimiento del siguiente texto de reseña de producto.

TEXTO A ANALIZAR:
"El producto llegó puntual pero la calidad es mediocre para el precio que cobran.
Esperaba mucho más basándome en las fotos de la web. El servicio de atención al
cliente fue amable cuando reclamé, así que algo es algo."

INSTRUCCIONES:
- Identifica el sentimiento general (positivo/negativo/mixto)
- Lista los aspectos positivos mencionados
- Lista los aspectos negativos mencionados
- Proporciona una puntuación del 1 al 5

FORMATO DE SALIDA: Responde en formato estructurado con secciones claramente separadas.
"""

In [17]:
# Invoca directamente con el prompt completo (sin template esta vez)
print(llm.invoke(human_completo).content)

### Prefijo forzado (AIMessage)

Una técnica avanzada: incluir el **inicio de la respuesta del modelo** para forzar un formato concreto.

Si le dices al modelo que ya ha empezado a responder con `'{"nombre":`, continuará en ese formato.

In [18]:
from langchain_core.messages import AIMessage

messages_con_prefijo = [
    SystemMessage(content="Eres un extractor de información. Siempre respondes en JSON válido."),
    HumanMessage(content="Extrae la información de este texto: 'Ana García, 32 años, vive en Madrid, trabaja como ingeniera de software en Telefónica desde 2019.'"),
    # Prefijo: le decimos al modelo que ya empezó a responder así, forzando el formato JSON
    AIMessage(content='{"nombre":')
]

In [19]:
resp = llm.invoke(messages_con_prefijo)
# El modelo continúa desde donde lo dejamos: completa el JSON
print(resp.content)

 "Ana García", "edad": 32, "ciudad": "Madrid", "profesion": "ingeniera de software", "empresa": "Telefónica", "fecha_inicio": "2019"}


## 10. Output Parsers (Respuestas Estructuradas)

Los **Output Parsers** convierten la respuesta del modelo (texto) en un **tipo de dato Python** utilizable.

### El problema sin parser:
```python
respuesta = llm.invoke("Dame info sobre Madrid en JSON")
respuesta.content  # → string: '{"ciudad": "Madrid", ...}'
                   # Necesitas parsearlo manualmente
```

### Con JsonOutputParser:
```python
cadena = template | llm | JsonOutputParser()
resultado = cadena.invoke(...)  # → dict Python directamente
resultado["ciudad"]  # → "Madrid"
```

In [20]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
import json

# JsonOutputParser convierte la respuesta JSON del modelo en un dict Python
parser_json = JsonOutputParser()

template_json = ChatPromptTemplate.from_messages([
    ("system", "Siempre responde con un JSON válido"),
    ("human", "{pregunta}")
])

# Cadena completa: template → llm → parser
# Cada | pasa la salida de un componente a la entrada del siguiente
cadena_json = template_json | llm | parser_json

resultado = cadena_json.invoke({
    "pregunta": "Dame informacion sobre Madrid: poblacion, pais, ubicacion, comida tipica"
})

# resultado ya es un dict Python, no un string
print(f"Tipo: {type(resultado).__name__}")
print(json.dumps(resultado, ensure_ascii=False, indent=2))

## 11. Pydantic (Validación Estricta)

**Pydantic** va un paso más allá de JSON: define un **schema exacto** con tipos y validaciones.

### ¿Por qué Pydantic y no JSON?
| JsonOutputParser | PydanticOutputParser |
|-----------------|---------------------|
| Devuelve un dict genérico | Devuelve un objeto Python con atributos tipados |
| No valida tipos | Valida que `puntuacion` sea int, no string |
| Si falta un campo, no hay error | Si falta un campo requerido, da error |
| Acceso: `resultado["campo"]` | Acceso: `resultado.campo` |

### Cuándo usar Pydantic:
- Cuando necesitas **garantizar** la estructura de la respuesta
- En producción, donde un error de formato puede romper el flujo

In [24]:
!pip install langchain

In [26]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from typing import List

# Define el schema: qué campos debe tener la respuesta y de qué tipo
class AnalisisSentimiento(BaseModel):
    # Field() añade metadatos: description le dice al modelo qué es cada campo
    clasificacion: str = Field(description="Clasificacion del sentimiento: POSITIVO, NEGATIVO o MIXTO")
    puntuacion: int = Field(description="Puntuacion del 1 al 5")                    # int → el modelo debe poner un número
    aspectos_negativos: List[str] = Field(description="Lista de aspectos negativos del texto")
    resumen: str = Field(description="Resumen breve de la reseña")
    recomendaria: bool = Field(description="Si el cliente recomendaria o no")       # bool → True/False

In [27]:
# PydanticOutputParser hace dos cosas:
# 1. Genera instrucciones de formato para incluir en el prompt (get_format_instructions())
# 2. Convierte la respuesta JSON del modelo en un objeto Python tipado
parser_pydantic = PydanticOutputParser(pydantic_object=AnalisisSentimiento)

template_pydantic = ChatPromptTemplate.from_messages([
    ("system", """Eres un analizador de reseñas de productos.

{format_instructions}"""),   # {format_instructions} se rellena con el schema JSON generado por el parser
    ("human", "Analiza esta reseña:\n\n{resena}")
])

# Cadena: template → llm → parser (que valida y convierte a objeto Python)
cadena_pydantic = template_pydantic | llm | parser_pydantic

In [28]:
RESENA_COMPLEJA = """
Compré este robot de cocina hace 3 meses y tengo sentimientos encontrados.
La potencia es brutal y pica verduras perfectamente, pero el ruido es ensordecedor.
El material parece de calidad y el diseño es elegante. Sin embargo, la app es un desastre
absoluto, se cuelga constantemente. El servicio técnico fue rapidísimo cuando reporté el problema.
Por el precio (450€) esperaba más, especialmente en software. El hardware, un 9; el software, un 3.
"""

resultado = cadena_pydantic.invoke({
    "resena": RESENA_COMPLEJA,
    "format_instructions": parser_pydantic.get_format_instructions()  # Schema JSON para el modelo
})

# Acceso como atributos de objeto (no como dict)
print(f"Clasificación: {resultado.clasificacion}")
print(f"Puntuación: {resultado.puntuacion}/5")
print(f"¿Recomendaría?: {'Sí' if resultado.recomendaria else 'No'}")
print(f"\nAspectos negativos:")
for asp in resultado.aspectos_negativos:
    print(f"  ❌ {asp}")
print(f"\nResumen: {resultado.resumen}")

## Resumen

| Concepto | Qué es | Cuándo usarlo |
|----------|--------|--------------|
| **LangChain** | Framework para LLMs | Siempre que construyas algo más que una llamada simple |
| **`invoke()`** | Una pregunta → una respuesta | Llamadas únicas |
| **`stream()`** | Respuesta token a token | Interfaces de usuario, respuestas largas |
| **`batch()`** | N preguntas en paralelo | Procesar lotes de datos |
| **Tools** | Funciones Python que el modelo puede llamar | Acceder a datos externos, APIs |
| **Template** | Prompt con variables reutilizables | Siempre que el prompt cambie con los datos |
| **Chain (`\|`)** | Encadenar componentes | Para conectar template → modelo → parser |
| **JsonOutputParser** | Convierte respuesta a dict | Cuando necesitas JSON sin schema estricto |
| **PydanticOutputParser** | Schema tipado y validado | Producción, cuando la estructura es crítica |